In [6]:
!pip install langchain faiss-cpu pandas sentence-transformers langdetect

     ---------------------------------------- 0.0/981.5 kB ? eta -:--:--
     ---------------------------------------- 0.0/981.5 kB ? eta -:--:--
     ---------- ----------------------------- 262.1/981.5 kB ? eta -:--:--
     ---------- ----------------------------- 262.1/981.5 kB ? eta -:--:--
     ---------------------------------------- 981.5/981.5 kB 1.7 MB/s  0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993364 sha256=94162227784a61b5ccb66eff3d4f1ae7c389eab2fe56ce4abb8ddb7c04008359
  Stored in directory: c:\users\hp\appdata\local\pip\cache\wheels\13\60\5a\f2012969b3f7413b62b7b7063d6c9b45c93004a978cbe8ea2f
Successful

# Étape                     |             Action
📥 Charger Excel         | Lire les questions/réponses
🤖 Embedding             | Encoder toutes les questions avec MiniLM
💬 Interaction           | L’utilisateur pose une question
🧠 Similarité            | Le bot cherche la question la plus proche
🗣 Réponse                | Il affiche la réponse associée

In [2]:
import pandas as pd
from sentence_transformers import SentenceTransformer, util

# Charger les données
df = pd.read_excel("chatbot_faq_9antra.xlsx")

# Charger un modèle local d'embedding
model = SentenceTransformer('all-MiniLM-L6-v2')

# Encoder les questions du fichier
questions = df['question'].tolist()
question_embeddings = model.encode(questions, convert_to_tensor=True)

# Boucle utilisateur
while True:
    user_question = input("Pose ta question (ou 'exit') : ")
    if user_question.lower() == "exit":
        break

    # Encoder la question de l'utilisateur
    user_embedding = model.encode(user_question, convert_to_tensor=True)

    # Mesurer la similarité
    scores = util.pytorch_cos_sim(user_embedding, question_embeddings)[0]
    best_match_idx = scores.argmax().item()
    best_score = scores[best_match_idx].item()

    # Afficher la réponse si la similarité est assez haute
    if best_score > 0.5:
        print(f"🤖 {df.iloc[best_match_idx]['reponse']} (score: {best_score:.2f})\n")
    else:
        print("🤖 Désolé, je ne trouve pas de réponse adaptée.\n")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Pose ta question (ou 'exit') :  quelle est la mission de 9antra ?


🤖 La mission de 9antra est de former les professionnels aux compétences de demain. (score: 1.00)



Pose ta question (ou 'exit') :  formation ?


🤖 Nous proposons des formations en développement web, marketing digital, design, et plus encore. (score: 0.58)



Pose ta question (ou 'exit') :  exit


In [3]:
!pip install deep-translator

In [4]:
Étape                    | Fonction
📥 Charger Excel         | Toutes les Q/R sont en français
🧠 Encoder les questions | Avec un modèle multilingue
💬 Détection de langue   | Avec langdetect
🔍 Recherche             | Trouver la question la plus proche
🌍 Traduction            | Si l’utilisateur n’est pas en français
🧾 Réponse               | Afficher ou traduire automatiquement

SyntaxError: invalid character '📥' (U+1F4E5) (442395127.py, line 2)

In [7]:
import pandas as pd
from sentence_transformers import SentenceTransformer, util
from deep_translator import GoogleTranslator
from langdetect import detect

# === Charger la FAQ en français ===
df = pd.read_excel("chatbot_faq_9antra.xlsx")  # fichier unique
questions = df['question'].tolist()
reponses = df['reponse'].tolist()

# === Embeddings multilingues ===
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
question_embeddings = model.encode(questions, convert_to_tensor=True)

print("\n🤖 Chatbot FAQ multilingue (1 fichier) — Tapez 'exit' pour quitter\n")

while True:
    user_input = input("👤 Vous : ")
    if user_input.lower() == "exit":
        print("👋 À bientôt !")
        break

    # Détection de la langue
    user_lang = detect(user_input)

    # Embedding & recherche
    user_embedding = model.encode(user_input, convert_to_tensor=True)
    scores = util.pytorch_cos_sim(user_embedding, question_embeddings)[0]
    best_idx = scores.argmax().item()
    best_score = scores[best_idx].item()

    if best_score > 0.5:
        reponse_fr = reponses[best_idx]
        # Traduction (si besoin)
        if user_lang != 'fr':
            try:
                reponse_traduite = GoogleTranslator(source='fr', target=user_lang).translate(reponse_fr)
                print(f"🤖 {reponse_traduite} (traduit du français)\n")
            except Exception as e:
                print(f"🤖 (FR) {reponse_fr} (⚠ erreur de traduction)\n")
        else:
            print(f"🤖 {reponse_fr}\n")
    else:
        print("🤖 Je ne trouve pas de réponse adaptée.\n")


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (8 outputs left out)

👤 Vous :  how to contact ?


🤖 (FR) Vous pouvez nous contacter par mail à contact@9antra.tn ou par téléphone au +216 71 000 000. (⚠ erreur de traduction)



👤 Vous :  exit


👋 À bientôt !


In [12]:
import pandas as pd
from sentence_transformers import SentenceTransformer, util
from deep_translator import GoogleTranslator
from langdetect import detect

# === Charger la FAQ ===
faq_df = pd.read_excel("chatbot_faq_9antra.xlsx")
faq_questions = faq_df['question'].tolist()
faq_reponses = faq_df['reponse'].tolist()

# === Charger les produits ===
produit_df = pd.read_excel("produits.xlsx")

# === Modèle multilingue ===
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
faq_embeddings = model.encode(faq_questions, convert_to_tensor=True)

def detect_lang(text):
    try:
        return detect(text)
    except:
        return 'fr'

def traduire_si_besoin(texte_fr, langue_user):
    if langue_user != 'fr':
        try:
            return GoogleTranslator(source='fr', target=langue_user).translate(texte_fr)
        except:
            return texte_fr + " (⚠ erreur de traduction)"
    return texte_fr

def reponse_faq(user_input, user_lang):
    user_embedding = model.encode(user_input, convert_to_tensor=True)
    scores = util.pytorch_cos_sim(user_embedding, faq_embeddings)[0]
    best_idx = scores.argmax().item()
    best_score = scores[best_idx].item()

    if best_score > 0.5:
        reponse_fr = faq_reponses[best_idx]
        # Traduction (si besoin)
        if user_lang != 'fr':
            try:
                reponse_traduite = GoogleTranslator(source='fr', target=user_lang).translate(reponse_fr)
                return f"{reponse_traduite} (traduit du français)"
            except Exception as e:
                return f"(FR) {reponse_fr} (⚠ erreur de traduction)"
        else:
            return reponse_fr
    return None


def reponse_produit(user_input, user_lang):
    # Traduire la question en français
    try:
        user_input_fr = GoogleTranslator(source='auto', target='fr').translate(user_input)
    except:
        user_input_fr = user_input

    text = user_input_fr.lower()

    # Chercher si un nom de produit est mentionné
    for index, row in produit_df.iterrows():
        nom_produit = row['nom'].lower()
        if nom_produit in text:
            # Identifier quelle information est demandée
            if "type" in text:
                reponse = f"Le type de {row['nom']} est : {row['type']}"
            elif "prix" in text or "coût" in text:
                reponse = f"Le prix de {row['nom']} est : {row['prix']}€"
            elif "note" in text:
                reponse = f"La note de {row['nom']} est : {row['note']}/5"
            elif "description" in text:
                reponse = f"La description de {row['nom']} est : {row['description']}"
            else:
                reponse = f"Voici les infos disponibles sur {row['nom']} :\n- Type : {row['type']}\n- Prix : {row['prix']}€\n- Note : {row['note']}/5\n- Description : {row['description']}"

            return traduire_si_besoin(reponse, user_lang)

    # Sinon, passer aux intentions générales (déjà gérées)
    return reponse_produit_par_intention(user_input_fr, user_lang)  # appel à la logique existante si besoin


# === Lancer le chatbot ===
print("\n🤖 Chatbot FAQ & Produits — Tapez 'exit' pour quitter\n")

while True:
    user_input = input("👤 Vous : ")
    if user_input.lower() == "exit":
        print("👋 À bientôt !")
        break

    lang = detect_lang(user_input)
    

    # Réponse via FAQ
    reponse = reponse_faq(user_input, lang)

    # Sinon, recherche dynamique
    if not reponse:
        reponse = reponse_produit(user_input, lang)

    # Si aucune réponse
    if not reponse:
        print("🤖 Je ne trouve pas de réponse adaptée.\n")
    else:
        print(f"🤖 {reponse}\n")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


🤖 Chatbot FAQ & Produits — Tapez 'exit' pour quitter



👤 Vous :  exit


👋 À bientôt !


In [13]:
!pip install ctransformers

   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   -------------------------- ------------- 6.6/9.9 MB 38.1 MB/s eta 0:00:01
   ---------------------------------------- 9.9/9.9 MB 34.9 MB/s  0:00:00


In [14]:
!pip install transformers accelerate torch

In [15]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# Tu peux changer de modèle ici 
model_id = "tiiuae/falcon-rw-1b"  # modèle léger et rapide

print("⏳ Chargement du modèle...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)

chatbot = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=200)

print("\n🤖 Chatbot généraliste prêt ! Tape 'exit' pour quitter.\n")

while True:
    user_input = input("👤 Vous : ")
    if user_input.lower() == "exit":
        print("👋 À bientôt !")
        break

    prompt = f"{user_input}\nRéponse :"
    response = chatbot(prompt)[0]['generated_text']
    
    # Afficher uniquement la réponse après le prompt
    reponse_finale = response.replace(prompt, "").strip()
    print("🤖", reponse_finale, "\n")


⏳ Chargement du modèle...


config.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

... (6 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.62GB            

model.safetensors: downloading bytes:           |  0.00B            

👤 Vous :  exit


👋 À bientôt !


In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, pipeline

model_id = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

chatbot = pipeline(
    "text2text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=100,
    temperature=0.7,
    top_p=0.9
)

print("\n🤖 Chatbot prêt avec Flan-T5 !")

while True:
    user_input = input("👤 Vous : ")
    if user_input.lower() == "exit":
        break

    # Flan-T5 fonctionne mieux avec des prompts directs
    prompt = f"Réponds à la question suivante : {user_input}"
    response = chatbot(prompt)[0]['generated_text']

    print("🤖", response.strip(), "\n")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

In [3]:
# === Sauvegarde du modèle (optionnel si tu veux éviter de le re-télécharger) ===
model.save("models/chatbot_model")

# === Sauvegarde des embeddings + data ===
import torch

torch.save({
    'questions': questions,
    'reponses': reponses,
    'embeddings': question_embeddings
}, 'models/faq_data.pt')


In [1]:
import nltk

# === Téléchargements nécessaires (1 seule fois) ===
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...


True

In [3]:
import pandas as pd
from sentence_transformers import SentenceTransformer, util
from deep_translator import GoogleTranslator
from langdetect import detect
import nltk
from nltk.corpus import wordnet

# === Charger les données ===
faq_df = pd.read_excel("chatbot_faq_9antra.xlsx")
faq_questions = faq_df['question'].tolist()
faq_reponses = faq_df['reponse'].tolist()

produit_df = pd.read_excel("produits.xlsx")

# === Modèle multilingue ===
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
faq_embeddings = model.encode(faq_questions, convert_to_tensor=True)

# === Fonctions utilitaires ===
def detect_lang(text):
    try:
        return detect(text)
    except:
        return 'fr'

def traduire_si_besoin(texte_fr, langue_user):
    if langue_user != 'fr':
        try:
            return GoogleTranslator(source='fr', target=langue_user).translate(texte_fr)
        except:
            return texte_fr + " (⚠ erreur de traduction)"
    return texte_fr

def traduire_vers_anglais(text):
    try:
        return GoogleTranslator(source='auto', target='en').translate(text)
    except:
        return text

def get_synonyms_wordnet(word):
    synonyms = set()
    for syn in wordnet.synsets(word, lang='eng'):
        for lemma in syn.lemmas():
            synonyms.add(lemma.name().replace('_', ' '))
    return list(synonyms)

def texte_contient_synonyme(text, mots_base):
    for mot in mots_base:
        syns = get_synonyms_wordnet(mot)
        syns.append(mot)
        for syn in syns:
            if syn in text:
                return True
    return False

# === 🔍 Recherche FAQ et Produits ===
def reponse_produit(user_input, user_lang):
    text = user_input.lower()
    text_en = traduire_vers_anglais(text)

    if texte_contient_synonyme(text_en, ['cheap', 'economical', 'affordable']):
        produit = produit_df.loc[produit_df['prix'].idxmin()]
        reponse = f"Le produit le moins cher est {produit['nom']} à {produit['prix']}€"
        return traduire_si_besoin(reponse, user_lang)

    elif texte_contient_synonyme(text_en, ['best restaurant']):
        restau = produit_df[produit_df['type'] == "restaurant"].sort_values(by="note", ascending=False).iloc[0]
        reponse = f"Le meilleur restaurant est {restau['nom']} avec une note de {restau['note']}/5"
        return traduire_si_besoin(reponse, user_lang)

    elif "5g" in text_en:
        produits_5g = produit_df[produit_df['description'].str.contains("5g", case=False)]
        if not produits_5g.empty:
            noms = ', '.join(produits_5g['nom'].tolist())
            reponse = f"Voici les produits 5G disponibles : {noms}"
        else:
            reponse = "Aucun produit 5G trouvé."
        return traduire_si_besoin(reponse, user_lang)

    return None

def reponse_faq(user_input, user_lang):
    question_emb = model.encode(user_input, convert_to_tensor=True)
    scores = util.pytorch_cos_sim(question_emb, faq_embeddings)[0]
    best_score = scores.max().item()
    best_idx = scores.argmax().item()

    if best_score > 0.6:  # seuil ajustable
        reponse = faq_reponses[best_idx]
        return traduire_si_besoin(reponse, user_lang)
    return None

# === 💬 Boucle du chatbot ===
print("\n🤖 Chatbot FAQ & Produits — Tapez 'exit' pour quitter\n")

while True:
    user_input = input("👤 Vous : ")
    if user_input.lower() == "exit":
        print("👋 À bientôt !")
        break

    lang = detect_lang(user_input)

    # Recherche FAQ
    reponse = reponse_faq(user_input, lang)

    # Sinon Produits
    if not reponse:
        reponse = reponse_produit(user_input, lang)

    # Réponse finale
    if not reponse:
        print("🤖 Je ne trouve pas de réponse adaptée.\n")
    else:
        print(f"🤖 {reponse}\n")



🤖 Chatbot FAQ & Produits — Tapez 'exit' pour quitter

👤 Vous : best restaurant
🤖 Le meilleur restaurant est Pizza Roma avec une note de 4.5/5

👤 Vous : cheap ?
🤖 The cheapest product is Burger King at 22 €

👤 Vous : meilleur restaurant 
🤖 Le meilleur restaurant est Pizza Roma avec une note de 4.5/5



KeyboardInterrupt: Interrupted by user